In [2]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL = "Qwen/Qwen3.5-2B"
device = "mps" if torch.backends.mps.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForMultimodalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(device)
model.eval()

messages = [{"role": "user", "content": [{"type": "text", "text": "Do mitochondria play a role in programmed cell death? Answer yes, no or maybe, then explain in one sentence."}]}]
inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                       return_dict=True, return_tensors="pt").to(device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=60, do_sample=False)

print(processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))


model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Yes, mitochondria play a central role in programmed cell death by releasing cytochrome c, which initiates the caspase cascade that leads to apoptosis.



In [3]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

with open("../data/corpus.json") as f:
    corpus = json.load(f)
with open("../data/questions.json") as f:
    questions = json.load(f)

doc_emb = np.load("../data/doc_emb.npy")
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def dense_search(query, k=3):
    q = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0]
    top = np.argsort(-(doc_emb @ q))[:k]
    return [corpus[i] for i in top]

q = questions[0]
docs = dense_search(q["question"])
print(q["question"])
print([d["doc_id"] for d in docs], "gold:", q["gold_doc_id"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
['21645374', '25430398', '25448003'] gold: 21645374


In [4]:
def build_messages(question, docs):
    if docs:
        context = "\n\n".join(f"[{i}] {d['text'][:1500]}" for i, d in enumerate(docs, start=1))
        text = (f"Abstracts:\n{context}\n\n"
                f"Question: {question}\n\n"
                "Using only the abstracts above, answer yes, no or maybe, "
                "then explain in one sentence citing the abstract number like [1].")
    else:
        text = (f"Question: {question}\n\n"
                "Answer yes, no or maybe, then explain in one sentence.")
    return [{"role": "user", "content": [{"type": "text", "text": text}]}]

def generate(messages, max_new_tokens=80):
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("GOLD:", q["label"])
print("\nNO RAG:", generate(build_messages(q["question"], [])))
print("\nRAG:   ", generate(build_messages(q["question"], docs)))

GOLD: yes

NO RAG: Yes, mitochondria play a crucial role in remodelling lace plant leaves during programmed cell death by releasing reactive oxygen species that trigger the downstream signaling pathways necessary for cell death.


RAG:    Yes, the abstract [1] suggests that mitochondria play a role in remodelling lace plant leaves during programmed cell death by delineating mitochondrial dynamics and examining the possible importance of mitochondrial permeability transition pore (PTP) formation during this process.



In [5]:
tokenizer = processor.tokenizer
LABELS = ["yes", "no", "maybe"]
label_ids = {lab: [tokenizer.encode(v, add_special_tokens=False)[0]
                   for v in (lab, lab.capitalize())]
             for lab in LABELS}

def classify(question, docs):
    msgs = build_messages(question, docs)
    msgs[0]["content"][0]["text"] = msgs[0]["content"][0]["text"].rsplit("\n\n", 1)[0] + \
        "\n\nReply with exactly one word: yes, no, or maybe."
    inputs = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1].float()
    probs = torch.softmax(logits, dim=-1)
    p = {lab: sum(probs[i].item() for i in ids) for lab, ids in label_ids.items()}
    total = sum(p.values())
    p = {lab: v / total for lab, v in p.items()}
    return max(p, key=p.get), p

print(classify(q["question"], []))
print(classify(q["question"], docs))

('yes', {'yes': 0.8617419582774074, 'no': 0.13483621413512273, 'maybe': 0.003421827587469879})
('yes', {'yes': 0.9671057494984904, 'no': 0.02675066994269966, 'maybe': 0.006143580558809999})


In [6]:
import time
from sklearn.metrics import accuracy_score, f1_score

id2doc = {d["doc_id"]: d for d in corpus}
N = 50

results = {"no_rag": [], "rag": [], "oracle": []}
gold = [q["label"] for q in questions[:N]]

start = time.time()
for i, q in enumerate(questions[:N]):
    results["no_rag"].append(classify(q["question"], [])[0])
    results["rag"].append(classify(q["question"], dense_search(q["question"], k=3))[0])
    results["oracle"].append(classify(q["question"], [id2doc[q["gold_doc_id"]]])[0])
    if i == 0:
        print(f"первый вопрос занял {time.time() - start:.1f} с")

print(f"всего: {time.time() - start:.0f} с\n")
for name, preds in results.items():
    print(f"{name:7} accuracy={accuracy_score(gold, preds):.3f}  "
          f"macro_f1={f1_score(gold, preds, average='macro'):.3f}")

первый вопрос занял 7.5 с
всего: 175 с

no_rag  accuracy=0.620  macro_f1=0.292
rag     accuracy=0.620  macro_f1=0.321
oracle  accuracy=0.680  macro_f1=0.413


In [7]:
from collections import Counter

print("gold:   ", Counter(gold))
for name, preds in results.items():
    print(f"{name:7}", Counter(preds))

gold:    Counter({'yes': 31, 'no': 15, 'maybe': 4})
no_rag  Counter({'yes': 48, 'no': 2})
rag     Counter({'yes': 46, 'no': 4})
oracle  Counter({'yes': 43, 'no': 7})


In [8]:
records = []
cf_no_rag = classify("N/A", [])[1]

start = time.time()
for i, q in enumerate(questions[:N]):
    rag_docs = dense_search(q["question"], k=3)
    oracle_docs = [id2doc[q["gold_doc_id"]]]
    records.append({
        "gold": q["label"],
        "no_rag": classify(q["question"], [])[1],
        "rag": classify(q["question"], rag_docs)[1],
        "oracle": classify(q["question"], oracle_docs)[1],
        "cf_no_rag": cf_no_rag,
        "cf_rag": classify("N/A", rag_docs)[1],
        "cf_oracle": classify("N/A", oracle_docs)[1],
    })
    if i == 0:
        print(f"первый вопрос: {time.time() - start:.1f} с")

print(f"всего: {time.time() - start:.0f} с")
print("bias без контекста:", {k: round(v, 3) for k, v in cf_no_rag.items()})

первый вопрос: 12.9 с
всего: 311 с
bias без контекста: {'yes': 0.797, 'no': 0.145, 'maybe': 0.059}


In [9]:
def calibrate(p, cf):
    adj = {lab: p[lab] / cf[lab] for lab in LABELS}
    total = sum(adj.values())
    return {lab: v / total for lab, v in adj.items()}

def argmax(p):
    return max(p, key=p.get)

gold = [r["gold"] for r in records]

for mode in ["no_rag", "rag", "oracle"]:
    raw = [argmax(r[mode]) for r in records]
    cal = [argmax(calibrate(r[mode], r[f"cf_{mode}"])) for r in records]
    for name, preds in [("raw", raw), ("calibrated", cal)]:
        print(f"{mode:7} {name:10} acc={accuracy_score(gold, preds):.3f}  "
              f"macro_f1={f1_score(gold, preds, average='macro'):.3f}  {dict(Counter(preds))}")
    print()

with open("../data/reader_records_50.json", "w") as f:
    json.dump(records, f)

no_rag  raw        acc=0.620  macro_f1=0.292  {'yes': 48, 'no': 2}
no_rag  calibrated acc=0.480  macro_f1=0.318  {'yes': 28, 'no': 20, 'maybe': 2}

rag     raw        acc=0.620  macro_f1=0.321  {'yes': 46, 'no': 4}
rag     calibrated acc=0.540  macro_f1=0.410  {'yes': 29, 'maybe': 17, 'no': 4}

oracle  raw        acc=0.680  macro_f1=0.413  {'yes': 43, 'no': 7}
oracle  calibrated acc=0.380  macro_f1=0.298  {'yes': 20, 'maybe': 26, 'no': 4}



In [10]:
for mode in ["no_rag", "rag", "oracle"]:
    cal = [argmax(calibrate(r[mode], r["cf_no_rag"])) for r in records]
    print(f"{mode:7} soft-cal  acc={accuracy_score(gold, cal):.3f}  "
          f"macro_f1={f1_score(gold, cal, average='macro'):.3f}  {dict(Counter(cal))}")
          

no_rag  soft-cal  acc=0.480  macro_f1=0.318  {'yes': 28, 'no': 20, 'maybe': 2}
rag     soft-cal  acc=0.580  macro_f1=0.417  {'yes': 33, 'maybe': 8, 'no': 9}
oracle  soft-cal  acc=0.640  macro_f1=0.505  {'yes': 28, 'maybe': 10, 'no': 12}


In [11]:
final = []
test_qs = questions[50:350]

start = time.time()
for i, q in enumerate(test_qs):
    final.append({
        "gold": q["label"],
        "no_rag": classify(q["question"], [])[1],
        "rag": classify(q["question"], dense_search(q["question"], k=3))[1],
        "oracle": classify(q["question"], [id2doc[q["gold_doc_id"]]])[1],
    })
    if (i + 1) % 25 == 0:
        with open("../data/reader_records_final.json", "w") as f:
            json.dump(final, f)
        print(f"{i + 1}/300  {time.time() - start:.0f} с")

with open("../data/reader_records_final.json", "w") as f:
    json.dump(final, f)
print("готово")


25/300  88 с
50/300  172 с
75/300  258 с
100/300  337 с
125/300  419 с
150/300  500 с
175/300  584 с
200/300  669 с
225/300  750 с
250/300  838 с
275/300  1150 с
300/300  1527 с
готово


In [12]:
gold = [r["gold"] for r in final]
majority = Counter(gold).most_common(1)[0][0]
maj_preds = [majority] * len(gold)

print("gold распределение:", dict(Counter(gold)))
print(f"majority ('{majority}')       acc={accuracy_score(gold, maj_preds):.3f}  "
      f"macro_f1={f1_score(gold, maj_preds, average='macro'):.3f}\n")

for mode in ["no_rag", "rag", "oracle"]:
    raw = [argmax(r[mode]) for r in final]
    cal = [argmax(calibrate(r[mode], cf_no_rag)) for r in final]
    for name, preds in [("raw", raw), ("soft-cal", cal)]:
        print(f"{mode:7} {name:9} acc={accuracy_score(gold, preds):.3f}  "
              f"macro_f1={f1_score(gold, preds, average='macro'):.3f}  {dict(Counter(preds))}")
    print()
    

gold распределение: {'maybe': 53, 'no': 86, 'yes': 161}
majority ('yes')       acc=0.537  macro_f1=0.233

no_rag  raw       acc=0.560  macro_f1=0.319  {'yes': 271, 'no': 29}
no_rag  soft-cal  acc=0.480  macro_f1=0.363  {'no': 129, 'yes': 160, 'maybe': 11}

rag     raw       acc=0.650  macro_f1=0.448  {'yes': 242, 'no': 58}
rag     soft-cal  acc=0.613  macro_f1=0.530  {'yes': 164, 'maybe': 49, 'no': 87}

oracle  raw       acc=0.690  macro_f1=0.498  {'yes': 229, 'no': 70, 'maybe': 1}
oracle  soft-cal  acc=0.620  macro_f1=0.551  {'yes': 146, 'no': 95, 'maybe': 59}

